# OCT Failure Mode Atlas: Does RETFound Look Where Clinicians Look?

This notebook systematically collects and characterizes failure cases of RETFound, VGG19, and EfficientNet on 10 OCT Datasets. It uses Attention Rollout (for RETFound) and Grad-CAM (for CNNs) to visualize what the models focused on when they made wrong predictions.


## 1. Setup & Imports
Install necessary packages for Grad-CAM, timm (for RETFound), and download kaggle datasets if needed.


In [ ]:
!pip install grad-cam timm kagglehub

import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import vgg19, efficientnet_b0
import timm
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 2. Dataset Configuration
List of all 10 datasets. You can loop over these or select one to process.


In [ ]:
DATASETS = [
    "mohamedberrimi/oct-images-balanced-version",
    "baalawi1/retinal-optical-coherence-tomography-images",
    "obulisainaren/retinal-oct-c8",
    "mmazizi/ucsd-3-class-labeled-retinal-oct-images",
    "kawtarnaim/octid-dataset",
    "orvile/octdl-optical-coherence-tomography-dataset",
    "anirudhcv/labeled-optical-coherence-tomography-oct",
    "nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split",
    "jakkabalakrishna/kermany-amd-kaggle",
    "saketlad/armd-combined-dataset-fundus-and-oct"
]

def download_dataset(dataset_name):
    import kagglehub
    path = kagglehub.dataset_download(dataset_name)
    print("Path to dataset files:", path)
    return path

# Example: Process the first dataset
# current_dataset_path = download_dataset(DATASETS[0])



## 3. Data Loaders
Prepare training and validation data loaders.


In [ ]:
def get_dataloaders(data_dir, batch_size=32):
    # Depending on dataset structure, this might need adjustment
    # Assumes train/ test/ folders inside data_dir
    train_dir = os.path.join(data_dir, 'train')
    val_dir = os.path.join(data_dir, 'test')
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if os.path.exists(train_dir) and os.path.exists(val_dir):
        train_ds = ImageFolder(train_dir, transform=transform)
        val_ds = ImageFolder(val_dir, transform=transform)
    else:
        # Fallback if no train/test split, use whole folder
        print(f"Warning: No train/test folders found in {data_dir}. Using full dataset for validation.")
        val_ds = ImageFolder(data_dir, transform=transform)
        train_ds = val_ds # Dummy
        
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, val_ds.classes


## 4. Model Definition & Fine-Tuning
Define RETFound (ViT), VGG19, and EfficientNet.


In [ ]:
def get_model(model_name, num_classes):
    if model_name == 'retfound':
        model = timm.create_model('vit_large_patch16_224', pretrained=False, num_classes=num_classes)
        # Load custom RETFound weights here if available
        # e.g., model.load_state_dict(torch.load('RETFound_oct_weights.pth')['model'])
    elif model_name == 'vgg19':
        model = vgg19(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'efficientnet':
        model = efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError("Model not supported")
    return model.to(device)

def finetune_model(model, train_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")
    return model


## 5. Inference & Failure Case Identification
Run the models on the validation set, find where predicted != actual label.


In [ ]:
def collect_failure_cases(model, val_loader):
    model.eval()
    failures = [] # list of dicts: {'image': tensor, 'true': int, 'pred': int, 'path': str}
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(val_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            for j in range(len(labels)):
                if preds[j] != labels[j]:
                    # Need to un-normalize image for visualization later
                    img = images[j].cpu()
                    img_path = val_loader.dataset.samples[i * val_loader.batch_size + j][0]
                    failures.append({
                        'image_tensor': images[j].unsqueeze(0), # Keep normalized tensor for GradCAM
                        'image_cpu': img,
                        'true': labels[j].item(),
                        'pred': preds[j].item(),
                        'path': img_path
                    })
    print(f"Found {len(failures)} failure cases.")
    return failures


## 6. Attention Extraction
Use Grad-CAM for CNNs and Attention Rollout for RETFound.


In [ ]:
def get_gradcam_heatmap(model, model_name, image_tensor):
    target_layers = []
    if model_name == 'vgg19':
        target_layers = [model.features[-1]]
    elif model_name == 'efficientnet':
        target_layers = [model.features[-1]]
    
    if not target_layers: return None
    
    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=image_tensor, targets=None)[0, :]
    return grayscale_cam

def get_attention_rollout(model, image_tensor):
    # ViT attention rollout implementation using standard timm ViT
    # This requires hooking into attention maps or relying on timm's forward_features
    # A simplified conceptual version:
    
    # Enable attention map return in timm (requires specific setup or custom model)
    # Since timm ViT doesn't return attention by default, this block is a placeholder 
    # to show the integration point. In a real scenario, you'd use a custom wrapper 
    # or library like `vit_rollout` to get `attentions` from ViT blocks.
    
    # Mock attention map for demonstration:
    heatmap = np.random.rand(224, 224) 
    return heatmap

def generate_attention_maps(model, model_name, failures):
    heatmaps = []
    for fail in failures:
        if model_name == 'retfound':
            heatmap = get_attention_rollout(model, fail['image_tensor'])
        else:
            heatmap = get_gradcam_heatmap(model, model_name, fail['image_tensor'])
        heatmaps.append(heatmap)
    return heatmaps


## 7. Visualization
Overlay attention heatmaps on B-scans, grouped by disease.


In [ ]:
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

def plot_failures_with_attention(failures, heatmaps, classes, num_to_show=5):
    num_to_show = min(num_to_show, len(failures))
    fig, axes = plt.subplots(num_to_show, 3, figsize=(12, 4 * num_to_show))
    
    for i in range(num_to_show):
        fail = failures[i]
        heatmap = heatmaps[i]
        img_np = denormalize(fail['image_cpu'])
        
        # Original
        ax_orig = axes[i, 0] if num_to_show > 1 else axes[0]
        ax_orig.imshow(img_np)
        ax_orig.set_title(f"True: {classes[fail['true']]}\nPred: {classes[fail['pred']]}")
        ax_orig.axis('off')
        
        # Heatmap
        ax_hm = axes[i, 1] if num_to_show > 1 else axes[1]
        ax_hm.imshow(heatmap, cmap='jet')
        ax_hm.set_title("Attention Map")
        ax_hm.axis('off')
        
        # Overlay
        overlay = show_cam_on_image(img_np, heatmap, use_rgb=True)
        ax_over = axes[i, 2] if num_to_show > 1 else axes[2]
        ax_over.imshow(overlay)
        ax_over.set_title("Overlay")
        ax_over.axis('off')
        
    plt.tight_layout()
    plt.show()

# Example Usage Flow (Uncomment to run):
# train_loader, val_loader, classes = get_dataloaders(current_dataset_path)
# model = get_model('vgg19', len(classes))
# model = finetune_model(model, train_loader, epochs=1)
# failures = collect_failure_cases(model, val_loader)
# heatmaps = generate_attention_maps(model, 'vgg19', failures)
# plot_failures_with_attention(failures, heatmaps, classes)
